# Inferential statiscs based on differnt probability distribution curves

# Critical Regions & p-values — interactive notebook

This notebook contains four interactive cells:
- **Standard Normal (Z)**
- **Student's t**
- **Chi-square (χ²)**
- **F-distribution**

Each cell lets you change:
- `alpha` (significance level),
- tail type: `left`, `right`, `two-sided`,
- distribution parameters (degrees of freedom where applicable),
- `sample_stat` (observed test statistic).

The cell will:
- draw the pdf,
- shade critical region(s),
- show numeric critical value(s),
- mark the sample statistic on the plot,
- print the computed p-value.

If widgets don't appear, make sure `ipywidgets` is enabled in your environment (e.g. `pip install ipywidgets` and `jupyter nbextension enable --py widgetsnbextension` for classic Jupyter).


# Environment setup


In [2]:
!pip install ipywidgets
# Cell 2 — Helper imports & functions
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from ipywidgets import interactive, FloatSlider, RadioButtons, IntSlider, FloatText, HBox, VBox
from IPython.display import display, Markdown

plt.rcParams['figure.figsize'] = (9,4)

def _shade_plot(x, pdf, crit_intervals, sample_stat=None, title=None):
    fig, ax = plt.subplots()
    ax.plot(x, pdf, linewidth=1.5)
    for (xl, xr) in crit_intervals:
        mask = (x >= xl) & (x <= xr)
        ax.fill_between(x[mask], 0, pdf[mask], alpha=0.6)
    if sample_stat is not None:
        ax.axvline(sample_stat, linestyle='--', linewidth=1.5, label=f"sample_stat = {sample_stat}")
        ax.legend()
    ax.set_xlabel("x")
    if title:
        ax.set_title(title)
    ax.grid(alpha=0.25)
    plt.show()

def _fmt_cv_list(cvs):
    return ", ".join([f"{c:.6g}" for c in cvs])


# Standard Normal (Z) Curve, with z-axis

In [4]:

def plot_normal(alpha=0.05, tail='two-sided', sample_stat=0.0):
    dist = stats.norm()
    x = np.linspace(-4, 4, 2000)
    pdf = dist.pdf(x)
    if tail == 'left':
        cv = dist.ppf(alpha)
        crit_intervals = [(-4, cv)]
        pval = dist.cdf(sample_stat)
        cvs = [cv]
    elif tail == 'right':
        cv = dist.ppf(1-alpha)
        crit_intervals = [(cv, 4)]
        pval = 1 - dist.cdf(sample_stat)
        cvs = [cv]
    else: # two-sided
        cv = dist.ppf(1 - alpha/2)
        crit_intervals = [(-4, -cv), (cv, 4)]
        pval = 2 * (1 - dist.cdf(abs(sample_stat)))
        cvs = [-cv, cv]

    display(Markdown(f"**Critical value(s):** {_fmt_cv_list(cvs)}  \n**p-value:** {pval:.6g}"))
    _shade_plot(x, pdf, crit_intervals, sample_stat, title=f"Standard Normal (alpha={alpha}, tail={tail})")

alpha_slider = FloatSlider(value=0.05, min=0.001, max=0.2, step=0.001, description='alpha')
tail_radio = RadioButtons(options=['left','right','two-sided'], value='two-sided', description='tail')
sample_input = FloatText(value=0.0, description='sample_stat')

w = interactive(plot_normal, alpha=alpha_slider, tail=tail_radio, sample_stat=sample_input)
display(w)


interactive(children=(FloatSlider(value=0.05, description='alpha', max=0.2, min=0.001, step=0.001), RadioButto…

# Student's t distribution , t-axis

In [5]:

def plot_t(alpha=0.05, tail='two-sided', df=10, sample_stat=0.0):
    dist = stats.t(df)
    x = np.linspace(dist.ppf(0.001), dist.ppf(0.999), 2000)
    pdf = dist.pdf(x)
    if tail == 'left':
        cv = dist.ppf(alpha)
        crit_intervals = [(x.min(), cv)]
        pval = dist.cdf(sample_stat)
        cvs = [cv]
    elif tail == 'right':
        cv = dist.ppf(1-alpha)
        crit_intervals = [(cv, x.max())]
        pval = 1 - dist.cdf(sample_stat)
        cvs = [cv]
    else:
        cv = dist.ppf(1 - alpha/2)
        crit_intervals = [(x.min(), -cv), (cv, x.max())]
        pval = 2 * (1 - dist.cdf(abs(sample_stat)))
        cvs = [-cv, cv]

    display(Markdown(f"**df:** {df}  \n**Critical value(s):** {_fmt_cv_list(cvs)}  \n**p-value:** {pval:.6g}"))
    _shade_plot(x, pdf, crit_intervals, sample_stat, title=f"Student's t (df={df}, alpha={alpha}, tail={tail})")

alpha_slider = FloatSlider(value=0.05, min=0.001, max=0.2, step=0.001, description='alpha')
tail_radio = RadioButtons(options=['left','right','two-sided'], value='two-sided', description='tail')
df_slider = IntSlider(value=10, min=1, max=500, step=1, description='df')
sample_input = FloatText(value=0.0, description='sample_stat')

w = interactive(plot_t, alpha=alpha_slider, tail=tail_radio, df=df_slider, sample_stat=sample_input)
display(w)


interactive(children=(FloatSlider(value=0.05, description='alpha', max=0.2, min=0.001, step=0.001), RadioButto…

# Chi-square distribution, chi-square axis

In [6]:
# Cell 5 — Chi-square — interactive
def plot_chi2(alpha=0.05, tail='right', df=10, sample_stat=0.0):
    dist = stats.chi2(df)
    x = np.linspace(0, dist.ppf(0.999), 2000)
    pdf = dist.pdf(x)
    if tail == 'left':
        cv = dist.ppf(alpha)
        crit_intervals = [(0, cv)]
        pval = dist.cdf(sample_stat)
        cvs = [cv]
    elif tail == 'right':
        cv = dist.ppf(1-alpha)
        crit_intervals = [(cv, x.max())]
        pval = 1 - dist.cdf(sample_stat)
        cvs = [cv]
    else:
        low = dist.ppf(alpha/2)
        high = dist.ppf(1-alpha/2)
        # two-sided chi2 is uncommon; we shade lower tail (0..low) and upper tail (high..max)
        crit_intervals = [(0, low), (high, x.max())]
        # p-value as two * min(tail probs), which is a sensible two-sided analogue
        pval = 2 * min(dist.cdf(sample_stat), 1 - dist.cdf(sample_stat))
        cvs = [low, high]

    display(Markdown(f"**df:** {df}  \n**Critical value(s):** {_fmt_cv_list(cvs)}  \n**p-value:** {pval:.6g}"))
    _shade_plot(x, pdf, crit_intervals, sample_stat, title=f"Chi-square (df={df}, alpha={alpha}, tail={tail})")

alpha_slider = FloatSlider(value=0.05, min=0.001, max=0.2, step=0.001, description='alpha')
tail_radio = RadioButtons(options=['left','right','two-sided'], value='right', description='tail')
df_slider = IntSlider(value=10, min=1, max=500, step=1, description='df')
sample_input = FloatText(value=0.0, description='sample_stat')

w = interactive(plot_chi2, alpha=alpha_slider, tail=tail_radio, df=df_slider, sample_stat=sample_input)
display(w)


interactive(children=(FloatSlider(value=0.05, description='alpha', max=0.2, min=0.001, step=0.001), RadioButto…

# F-distribution , f-axis

In [7]:

def plot_f(alpha=0.05, tail='right', dfn=5, dfd=10, sample_stat=0.0):
    dist = stats.f(dfn, dfd)
    x = np.linspace(0, dist.ppf(0.995), 2000)
    pdf = dist.pdf(x)
    if tail == 'left':
        cv = dist.ppf(alpha)
        crit_intervals = [(0, cv)]
        pval = dist.cdf(sample_stat)
        cvs = [cv]
    elif tail == 'right':
        cv = dist.ppf(1-alpha)
        crit_intervals = [(cv, x.max())]
        pval = 1 - dist.cdf(sample_stat)
        cvs = [cv]
    else:
        low = dist.ppf(alpha/2)
        high = dist.ppf(1-alpha/2)
        crit_intervals = [(0, low), (high, x.max())]
        pval = 2 * min(dist.cdf(sample_stat), 1 - dist.cdf(sample_stat))
        cvs = [low, high]

    display(Markdown(f"**dfn (num):** {dfn}, **dfd (den):** {dfd}  \n**Critical value(s):** {_fmt_cv_list(cvs)}  \n**p-value:** {pval:.6g}"))
    _shade_plot(x, pdf, crit_intervals, sample_stat, title=f"F (dfn={dfn}, dfd={dfd}, alpha={alpha}, tail={tail})")

alpha_slider = FloatSlider(value=0.05, min=0.001, max=0.2, step=0.001, description='alpha')
tail_radio = RadioButtons(options=['left','right','two-sided'], value='right', description='tail')
dfn_slider = IntSlider(value=5, min=1, max=500, step=1, description='dfn')
dfd_slider = IntSlider(value=10, min=1, max=500, step=1, description='dfd')
sample_input = FloatText(value=0.0, description='sample_stat')

w = interactive(plot_f, alpha=alpha_slider, tail=tail_radio, dfn=dfn_slider, dfd=dfd_slider, sample_stat=sample_input)
display(w)


interactive(children=(FloatSlider(value=0.05, description='alpha', max=0.2, min=0.001, step=0.001), RadioButto…